In [ ]:
# ============================================================
# CELL 1: Setup
# ============================================================

import pandas as pd
import numpy as np
import time
import warnings
from datetime import timedelta
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import pipeline
import torch

# Local data directory (use INF2006_Data_Students folder)
DATA_DIR = Path(__file__).parent / 'INF2006_Data_Students' if '__file__' in dir() else Path('.') / 'INF2006_Data_Students'
# Fallback: set explicitly if the above doesn't resolve correctly
if not DATA_DIR.exists():
    DATA_DIR = Path(r'c:\Users\Sebert\Desktop\SIT Lambert\Y2T2\INF2006 - Cloud Computing and Big Data\Project\Algorithmic Trading\INF2006_Data_Students')

warnings.filterwarnings('ignore')

# Configuration
STARTING_CASH = 100000
FINBERT_MODEL = "ProsusAI/finbert"
DEVICE = 0 if torch.cuda.is_available() else -1

print(f"Data directory: {DATA_DIR}")
print(f"Initial capital: ${STARTING_CASH:,.0f}")
print(f"Device: {'GPU' if DEVICE >= 0 else 'CPU'}")

In [ ]:
# ============================================================
# CELL 2: Initialize FinBERT
# ============================================================

print("Loading FinBERT (first run downloads ~420MB)...")

finbert_pipeline = pipeline(
    "sentiment-analysis",
    model=FINBERT_MODEL,
    tokenizer=FINBERT_MODEL,
    device=DEVICE,
    return_all_scores=True,
    truncation=True,
    max_length=512
)

print("FinBERT loaded.")

In [ ]:
# ============================================================
# CELL 3: Data Loading
# ============================================================

import shutil
import tempfile

def load_prices(split='dev'):
    filepath = DATA_DIR / f'prices_{split}.parquet'
    if not filepath.exists():
        raise FileNotFoundError(f"Not found: {filepath}")
    # Copy to short temp path to avoid Windows long-path issues
    tmp = Path(tempfile.gettempdir()) / f'prices_{split}.parquet'
    shutil.copy2(filepath, tmp)
    return pd.read_parquet(tmp)

def load_earnings(split='dev'):
    filepath = DATA_DIR / f'earnings_{split}.parquet'
    if not filepath.exists():
        raise FileNotFoundError(f"Not found: {filepath}")
    tmp = Path(tempfile.gettempdir()) / f'earnings_{split}.parquet'
    shutil.copy2(filepath, tmp)
    return pd.read_parquet(tmp)

prices_dev = load_prices('dev')
prices_val = load_prices('val')
earnings_dev = load_earnings('dev')
earnings_val = load_earnings('val')

print(f"Dev prices:   {len(prices_dev):,} records")
print(f"Val prices:   {len(prices_val):,} records")
print(f"Dev earnings: {len(earnings_dev):,} records")
print(f"Val earnings: {len(earnings_val):,} records")

In [ ]:
# ============================================================
# CELL 4: Exploratory Data Analysis
# ============================================================

print("="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

print(f"\n[PRICE DATA]")
print(f"Date range: {prices_dev['date'].min()} to {prices_dev['date'].max()}")
print(f"Unique tickers: {prices_dev['ticker'].nunique()}")
print(f"\nPrice statistics:")
print(prices_dev[['open', 'high', 'low', 'close', 'volume']].describe())

print(f"\n[MISSING VALUES]")
print(prices_dev.isnull().sum())

print(f"\n[EARNINGS DATA]")
print(f"Date range: {earnings_dev['date'].min()} to {earnings_dev['date'].max()}")
print(f"Unique tickers: {earnings_dev['ticker'].nunique()}")

earnings_dev['transcript_length'] = earnings_dev['transcript'].str.len()
print(f"\nTranscript lengths:")
print(earnings_dev['transcript_length'].describe())

price_tickers = set(prices_dev['ticker'].unique())
earnings_tickers = set(earnings_dev['ticker'].unique())
overlap = price_tickers & earnings_tickers
print(f"\n[TICKER OVERLAP]")
print(f"Prices: {len(price_tickers)}, Earnings: {len(earnings_tickers)}, Overlap: {len(overlap)}")

print("="*60)

In [ ]:
from pathlib import Path
EDA_PLOTS_DIR = Path('eda_plots')
EDA_PLOTS_DIR.mkdir(exist_ok=True)
print(f'EDA plots will be saved to: {EDA_PLOTS_DIR.resolve()}')

# ============================================================
# CELL 5: Data Visualization
# ============================================================

sample_ticker = prices_dev['ticker'].value_counts().index[0]
sample_prices = prices_dev[prices_dev['ticker'] == sample_ticker].sort_values('date')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(pd.to_datetime(sample_prices['date']), sample_prices['close'])
axes[0, 0].set_title(f'{sample_ticker} Price Over Time')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Close Price')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(pd.to_datetime(sample_prices['date']), sample_prices['volume'])
axes[0, 1].set_title(f'{sample_ticker} Volume')
axes[0, 1].set_xlabel('Date')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(prices_dev['close'], bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Price Distribution')
axes[1, 0].set_xlabel('Close Price')
axes[1, 0].grid(True, alpha=0.3)

ticker_counts = prices_dev['ticker'].value_counts().head(20)
axes[1, 1].barh(range(len(ticker_counts)), ticker_counts.values)
axes[1, 1].set_yticks(range(len(ticker_counts)))
axes[1, 1].set_yticklabels(ticker_counts.index)
axes[1, 1].set_title('Top 20 Tickers by Data Points')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
try:
    plt.savefig(EDA_PLOTS_DIR / 'eda_00_overview.png', dpi=150, bbox_inches='tight')
    print('Saved eda_00_overview.png')
except Exception as _e:
    print('SAVE FAILED: ' + str(_e))
plt.show()

# Enchanced EDA - Data Exploration

In [ ]:
from pathlib import Path
EDA_PLOTS_DIR = Path('eda_plots')
EDA_PLOTS_DIR.mkdir(exist_ok=True)
print(f'EDA plots will be saved to: {EDA_PLOTS_DIR.resolve()}')

# ============================================================
# CELL 12: Additional EDA  (VECTORISED)
# ============================================================
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from statsmodels.tsa.stattools import ccf
from statsmodels.stats.diagnostic import acorr_ljungbox
import networkx as nx
from tqdm import tqdm
import re

# ============================================================
# Earnings Call Transcript Availability
# ============================================================
print("\n[EARNINGS TRANSCRIPTS] Listing all available transcripts sorted by ticker and date...")

earnings_list = earnings_dev[['ticker', 'date', 'quarter']].copy()
earnings_list['date'] = pd.to_datetime(earnings_list['date']).dt.tz_localize(None)
earnings_list = earnings_list.sort_values(['ticker', 'date']).reset_index(drop=True)
earnings_list['transcript_length'] = earnings_dev['transcript'].str.len().values

print(f"\nTotal transcripts available: {len(earnings_list):,}")
print(f"Unique tickers with transcripts: {earnings_list['ticker'].nunique()}")
print(f"Date range: {earnings_list['date'].min().date()} to {earnings_list['date'].max().date()}")

print("\n" + "="*70)
print(f"{'#':<6} {'Ticker':<10} {'Date':<14} {'Quarter':<12} {'Length':>10}")
print("="*70)

for i, row in earnings_list.iterrows():
    print(f"{i+1:<6} {row['ticker']:<10} {str(row['date'].date()):<14} {str(row.get('quarter', 'N/A')):<12} {row['transcript_length']:>10,}")

print("="*70)

# Summary per ticker
print("\n[PER-TICKER SUMMARY]")
summary = earnings_list.groupby('ticker').agg(
    count=('date', 'count'),
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    avg_length=('transcript_length', 'mean')
).reset_index().sort_values('ticker')

print(f"\n{'Ticker':<10} {'Count':>6} {'First Date':<14} {'Last Date':<14} {'Avg Length':>12}")
print("-"*60)
for _, row in summary.iterrows():
    print(f"{row['ticker']:<10} {row['count']:>6} {str(row['first_date'].date()):<14} {str(row['last_date'].date()):<14} {row['avg_length']:>12,.0f}")


# ============================================================
# Interactive Price Time Series — All Tickers (vectorised normalisation)
# ============================================================
print("\n[PRICE TIME SERIES] Plotting normalised interactive price chart for all tickers...")

prices_ts = prices_dev.copy()
prices_ts['date'] = pd.to_datetime(prices_ts['date']).dt.tz_localize(None)
prices_ts = prices_ts.sort_values(['ticker', 'date'])

# Vectorised normalisation: divide by first close per ticker
prices_ts['first_close'] = prices_ts.groupby('ticker')['close'].transform('first')
prices_ts['normalised'] = (prices_ts['close'] / prices_ts['first_close']) * 100
prices_ts = prices_ts[prices_ts['first_close'] > 0]

all_tickers = sorted(prices_ts['ticker'].unique())

fig = go.Figure()

for ticker in all_tickers:
    mask = prices_ts['ticker'] == ticker
    tdf = prices_ts.loc[mask]
    fig.add_trace(go.Scattergl(
        x=tdf['date'],
        y=tdf['normalised'],
        mode='lines',
        name=ticker,
        line=dict(width=1),
        visible='legendonly',
        hovertemplate=(
            f'<b>{ticker}</b><br>'
            'Date: %{x|%Y-%m-%d}<br>'
            'Indexed: %{y:.1f}<br>'
            'Close: $%{customdata:.2f}'
            '<extra></extra>'
        ),
        customdata=tdf['close'].values
    ))

fig.add_hline(y=100, line=dict(color='black', dash='dash', width=1),
              annotation_text='Base (100)', annotation_position='bottom right')

fig.update_layout(
    title='S&P 500 Normalised Price Paths (Dev Split) — Base 100 at first trading day',
    xaxis_title='Date', yaxis_title='Normalised Price (Base = 100)',
    height=650, width=1400, hovermode='closest',
    legend=dict(orientation='v', x=1.01, y=1, font=dict(size=9),
                itemclick='toggle', itemdoubleclick='toggleothers'),
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(buttons=[
            dict(count=1, label='1Y', step='year', stepmode='backward'),
            dict(count=3, label='3Y', step='year', stepmode='backward'),
            dict(count=5, label='5Y', step='year', stepmode='backward'),
            dict(count=10, label='10Y', step='year', stepmode='backward'),
            dict(step='all', label='All')
        ])
    ),
    yaxis=dict(fixedrange=False),
    updatemenus=[dict(type='buttons', direction='left', x=0.0, y=1.12, buttons=[
        dict(label='Show All', method='restyle', args=[{'visible': True}]),
        dict(label='Hide All', method='restyle', args=[{'visible': 'legendonly'}])
    ])]
)
try:
    fig.write_image(EDA_PLOTS_DIR / 'eda_01_normalized_price_chart.png', scale=2)
    print('Saved eda_01_normalized_price_chart.png')
except Exception as _e:
    print('SAVE FAILED eda_01_normalized_price_chart.png: ' + str(_e))
fig.show()
print(f"Normalised chart rendered with {len(all_tickers)} tickers.")

# --------------- Helpers ---------------
prices_eda = prices_dev.copy()
prices_eda['date'] = pd.to_datetime(prices_eda['date']).dt.tz_localize(None)
prices_eda = prices_eda.sort_values(['ticker', 'date'])

earnings_eda = earnings_dev.copy()
earnings_eda['date'] = pd.to_datetime(earnings_eda['date']).dt.tz_localize(None)

top30 = prices_eda['ticker'].value_counts().head(30).index.tolist()
print(f"Top 30 tickers for EDA: {top30}")

# Vectorised log returns
prices_eda['log_return'] = prices_eda.groupby('ticker')['close'].transform(
    lambda x: np.log(x / x.shift(1))
)

# ============================================================
# DATA AVAILABILITY — SINGLE TRACE (vectorised)
# ============================================================
print("\n[1] Plotting data availability for all tickers...")

all_tickers_sorted = sorted(prices_eda['ticker'].unique())

fig = go.Figure()
# >>> VECTORISED: one single Scattergl trace instead of ~400 <<<
fig.add_trace(go.Scattergl(
    x=prices_eda['date'].values,
    y=prices_eda['ticker'].values,
    mode='markers',
    marker=dict(size=1.5, color='steelblue', opacity=0.6),
    showlegend=False,
    hovertemplate='%{y}<br>Date: %{x}<extra></extra>'
))

fig.update_layout(
    title='Price Data Availability by Ticker (gaps indicate delisting / missing data)',
    xaxis_title='Date', yaxis_title='Ticker',
    height=max(600, len(all_tickers_sorted) * 8), width=1200,
    yaxis=dict(tickfont=dict(size=6), categoryorder='array',
               categoryarray=all_tickers_sorted)
)
try:
    fig.write_image(EDA_PLOTS_DIR / 'eda_02_data_availability.png', scale=2)
    print('Saved eda_02_data_availability.png')
except Exception as _e:
    print('SAVE FAILED eda_02_data_availability.png: ' + str(_e))
fig.show()

# ============================================================
# LJUNG-BOX EFFICIENCY TEST — VECTORISED via groupby
# ============================================================
print("\n[EDA] Computing Ljung-Box test (lag=20) on log returns for all tickers...")

LAG_TO_TEST = 20

lb_results = {}
for ticker, group in prices_eda.groupby('ticker')['log_return']:
    lr = group.dropna()
    if len(lr) < 50:
        continue
    try:
        res = acorr_ljungbox(lr, lags=[LAG_TO_TEST], return_df=True)
        lb_results[ticker] = {
            'lb_pvalue': res['lb_pvalue'].iloc[0],
            'lb_stat':   res['lb_stat'].iloc[0]
        }
    except Exception:
        continue

lb_df = (pd.DataFrame.from_dict(lb_results, orient='index')
         .rename_axis('ticker')
         .reset_index()
         .sort_values('lb_pvalue')
         .reset_index(drop=True))

alpha = 0.05
n_total = len(lb_df)
n_inefficient = (lb_df['lb_pvalue'] < alpha).sum()
n_efficient = n_total - n_inefficient

print(f"\nTotal tickers tested: {n_total}")
print(f"Inefficient (p < {alpha}): {n_inefficient}  ({n_inefficient/n_total:.1%})")
print(f"Efficient   (p >= {alpha}): {n_efficient}  ({n_efficient/n_total:.1%})")

colors = np.where(lb_df['lb_pvalue'] < alpha, '#E15759', '#76B7B2')

fig_lb = go.Figure()
fig_lb.add_trace(go.Bar(
    x=lb_df['ticker'], y=lb_df['lb_pvalue'], marker_color=colors, showlegend=False,
    hovertemplate='<b>%{x}</b><br>Ljung-Box p-value: %{y:.4e}<br>Q-Stat: %{customdata:.2f}<extra></extra>',
    customdata=lb_df['lb_stat']
))
fig_lb.add_hline(y=alpha, line=dict(color='black', width=2, dash='dash'),
                 annotation_text=f'α = {alpha}', annotation_position='top right',
                 annotation_font=dict(size=14, color='black'))
fig_lb.add_trace(go.Bar(x=[None], y=[None], marker_color='#E15759',
                        name=f'Inefficient / Tradable p < {alpha}  (n={n_inefficient})', showlegend=True))
fig_lb.add_trace(go.Bar(x=[None], y=[None], marker_color='#76B7B2',
                        name=f'Efficient / Random Walk p ≥ {alpha}  (n={n_efficient})', showlegend=True))
fig_lb.update_layout(
    title=(f'Ljung-Box Test for Autocorrelation — {n_total} Tickers<br>'
           f'<sup>Hypothesis: Stocks below the dashed line (p < {alpha}) exhibit serial correlation and reject the random walk.</sup>'),
    xaxis_title='Ticker (sorted by p-value)',
    yaxis_title=f'Ljung-Box p-value (Lag = {LAG_TO_TEST})',
    height=600, width=1400,
    xaxis=dict(tickfont=dict(size=6), tickangle=90,
               categoryorder='array', categoryarray=lb_df['ticker'].tolist()),
    yaxis=dict(type='log', title='p-value (Log Scale)'),
    legend=dict(x=0.01, y=0.98, font=dict(size=12),
                bgcolor='rgba(255,255,255,0.9)', bordercolor='black', borderwidth=1),
    plot_bgcolor='white', hovermode='closest'
)
try:
    fig_lb.write_image(EDA_PLOTS_DIR / 'eda_03_ljung_box.png', scale=2)
    print('Saved eda_03_ljung_box.png')
except Exception as _e:
    print('SAVE FAILED eda_03_ljung_box.png: ' + str(_e))
fig_lb.show()

LB_INEFFICIENT_TICKERS = set(lb_df[lb_df['lb_pvalue'] < alpha]['ticker'].tolist())
print(f"\nStored as LB_INEFFICIENT_TICKERS (set of {len(LB_INEFFICIENT_TICKERS)} tickers)")

INEFFICIENT_TICKERS = LB_INEFFICIENT_TICKERS
print(f"\nUsing INEFFICIENT_TICKERS from Ljung-Box (set of {len(INEFFICIENT_TICKERS)} tickers)")

# ============================================================
# CORRELATION MATRIX HEATMAP — INEFFICIENT TICKERS ONLY
# ============================================================
print("\n[9] Computing return correlation matrix for INEFFICIENT tickers only...")

returns_pivot = prices_eda.pivot_table(index='date', columns='ticker', values='log_return')
min_obs = 252
valid_tickers = returns_pivot.columns[returns_pivot.notna().sum() >= min_obs]
returns_pivot = returns_pivot[valid_tickers]

inefficient_valid = [t for t in valid_tickers if t in INEFFICIENT_TICKERS]
returns_pivot = returns_pivot[inefficient_valid]
corr_matrix = returns_pivot.corr()

print(f"Filtered from {len(valid_tickers)} → {len(inefficient_valid)} inefficient tickers")

# ================================================================
# HIERARCHICAL CLUSTERING (Ward)
# ================================================================
print("\n[10] Hierarchical clustering on inefficient ticker correlations...")

corr_clean = corr_matrix.copy()
corr_clean_vals = corr_clean.to_numpy(copy=True, dtype=float)
np.fill_diagonal(corr_clean_vals, 1.0)
corr_clean = pd.DataFrame(corr_clean_vals, index=corr_clean.index, columns=corr_clean.columns)
corr_clean = corr_clean.clip(-1, 1)
dist_matrix = 1 - corr_clean
dist_condensed = squareform(dist_matrix.values, checks=False)
Z = linkage(dist_condensed, method='ward')

dendro_data = dendrogram(Z, labels=corr_matrix.columns.tolist(),
                         no_plot=True, color_threshold=0.7 * max(Z[:, 2]))

for n_clusters in [5, 10, 15]:
    labels = fcluster(Z, t=n_clusters, criterion='maxclust')
    cluster_df = pd.DataFrame({'ticker': corr_matrix.columns, 'cluster': labels})
    print(f"\n--- {n_clusters} Clusters ---")
    for c in sorted(cluster_df['cluster'].unique()):
        members = cluster_df[cluster_df['cluster'] == c]['ticker'].tolist()
        print(f"  Cluster {c} ({len(members):>3} tickers): {', '.join(members[:15])}"
              f"{'...' if len(members) > 15 else ''}")

# Reordered heatmap
print("\nPlotting cluster-ordered correlation heatmap (inefficient tickers only)...")
reorder_idx = dendro_data['leaves']
reordered_corr = corr_matrix.iloc[reorder_idx, reorder_idx]

fig_heatmap = go.Figure(data=go.Heatmap(
    z=reordered_corr.values,
    x=reordered_corr.columns.tolist(),
    y=reordered_corr.index.tolist(),
    colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
    colorbar=dict(title='ρ'),
    hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>'
))
fig_heatmap.update_layout(
    title=f'Correlation Heatmap — Inefficient Tickers Only ({len(inefficient_valid)}) — Ordered by Hierarchical Clustering',
    height=950, width=1000,
    xaxis=dict(tickfont=dict(size=6), tickangle=90),
    yaxis=dict(tickfont=dict(size=6), autorange='reversed')
)
try:
    fig_heatmap.write_image(EDA_PLOTS_DIR / 'eda_04_correlation_heatmap.png', scale=2)
    print('Saved eda_04_correlation_heatmap.png')
except Exception as _e:
    print('SAVE FAILED eda_04_correlation_heatmap.png: ' + str(_e))
fig_heatmap.show()

# ================================================================
# PRE-COMPUTE AVERAGE DAILY VOLUME
# ================================================================
print("\nComputing average daily volume per ticker for leader identification...")
avg_volume = (
    prices_eda[prices_eda['ticker'].isin(INEFFICIENT_TICKERS)]
    .groupby('ticker')['volume'].mean().to_dict()
)

# ================================================================
# LOUVAIN COMMUNITY DETECTION — VECTORISED edge building
# ================================================================
print("\n[11] Louvain community detection (INEFFICIENT tickers only, ρ ≥ 0.6)...")

CORR_THRESHOLD = 0.6

G = nx.Graph()
tickers_list = corr_matrix.columns.tolist()
G.add_nodes_from(tickers_list)

# >>> VECTORISED: NumPy upper-triangle mask instead of O(n²) Python loop <<<
corr_vals = corr_matrix.values
row_idx, col_idx = np.triu_indices(len(tickers_list), k=1)
rho_vals = corr_vals[row_idx, col_idx]
edge_mask = rho_vals >= CORR_THRESHOLD

edges = [
    (tickers_list[r], tickers_list[c], {'weight': float(rho)})
    for r, c, rho in zip(row_idx[edge_mask], col_idx[edge_mask], rho_vals[edge_mask])
]
G.add_edges_from(edges)
edge_count = len(edges)

print(f"Graph: {G.number_of_nodes()} nodes, {edge_count} edges (ρ ≥ {CORR_THRESHOLD})")

# Remove isolated nodes
isolated = [n for n in G.nodes() if G.degree(n) == 0]
G_connected = G.copy()
G_connected.remove_nodes_from(isolated)
print(f"After removing {len(isolated)} isolated nodes: {G_connected.number_of_nodes()} nodes, {G_connected.number_of_edges()} edges")

# Louvain communities
communities = nx.community.louvain_communities(G_connected, weight='weight', seed=42)
print(f"Detected {len(communities)} communities")

# Leader of each community by average volume
community_leaders = {}
for idx, comm in enumerate(sorted(communities, key=len, reverse=True)):
    members = sorted(comm)
    leader = max(members, key=lambda t: avg_volume.get(t, 0))
    leader_vol = avg_volume.get(leader, 0)
    community_leaders[idx] = leader
    print(f"  Community {idx} ({len(members):>3} tickers) | "
          f"Leader: {leader} (avg vol: {leader_vol:,.0f}) | "
          f"Members: {', '.join(members[:18])}"
          f"{'...' if len(members) > 18 else ''}")

# ---------- Interactive network visualisation ----------
print("\nComputing network layout (spring layout)...")

pos = nx.spring_layout(G_connected, k=1.5 / np.sqrt(G_connected.number_of_nodes()),
                        iterations=80, seed=42, weight='weight')

palette = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
community_colors = {i: palette[i % len(palette)] for i in range(len(communities))}

# Edge traces (already vectorised — list concat)
edge_x, edge_y = [], []
for u, v, data in G_connected.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

fig_net = go.Figure()
fig_net.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.3, color='#cccccc'),
    hoverinfo='none', showlegend=False
))

leader_set = set(community_leaders.values())

for comm_idx, comm in enumerate(sorted(communities, key=len, reverse=True)):
    comm_nodes = sorted(comm)
    leader = community_leaders[comm_idx]

    regular = [n for n in comm_nodes if n != leader]
    if regular:
        node_x = [pos[n][0] for n in regular]
        node_y = [pos[n][1] for n in regular]
        node_sizes = [5 + G_connected.degree(n) * 1.5 for n in regular]
        hover_texts = [
            f"<b>{n}</b><br>Community: {comm_idx}<br>"
            f"Degree: {G_connected.degree(n)}<br>"
            f"Avg Volume: {avg_volume.get(n, 0):,.0f}<br>"
            f"Connections: {', '.join(sorted(G_connected.neighbors(n)))}"
            for n in regular
        ]
        fig_net.add_trace(go.Scatter(
            x=node_x, y=node_y, mode='markers+text',
            marker=dict(size=node_sizes, color=community_colors[comm_idx],
                        line=dict(width=0.5, color='white'), symbol='circle'),
            text=regular, textposition='top center', textfont=dict(size=7),
            hovertext=hover_texts, hoverinfo='text',
            name=f'Community {comm_idx} ({len(comm_nodes)})', showlegend=True
        ))

    fig_net.add_trace(go.Scatter(
        x=[pos[leader][0]], y=[pos[leader][1]], mode='markers+text',
        marker=dict(size=20 + G_connected.degree(leader) * 2,
                    color=community_colors[comm_idx],
                    line=dict(width=2, color='black'), symbol='star'),
        text=[f'★ {leader}'], textposition='top center',
        textfont=dict(size=10, color='black'),
        hovertext=[
            f"<b>★ LEADER: {leader}</b><br>Community: {comm_idx}<br>"
            f"Degree: {G_connected.degree(leader)}<br>"
            f"Avg Volume: {avg_volume.get(leader, 0):,.0f}<br>"
            f"Connections: {', '.join(sorted(G_connected.neighbors(leader)))}"
        ],
        hoverinfo='text', name=f'★ Leader: {leader}', showlegend=True
    ))

fig_net.update_layout(
    title=f'Louvain Community Network — INEFFICIENT Tickers Only (ρ ≥ {CORR_THRESHOLD})<br>'
          f'<sub>{G_connected.number_of_nodes()} stocks, {G_connected.number_of_edges()} edges, '
          f'{len(communities)} communities — ★ = Leader by Avg Daily Volume</sub>',
    height=850, width=1200,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    hovermode='closest',
    legend=dict(title='Communities & Leaders', font=dict(size=10),
                itemclick='toggle', itemdoubleclick='toggleothers'),
    plot_bgcolor='white'
)
try:
    fig_net.write_image(EDA_PLOTS_DIR / 'eda_05_community_network.png', scale=2)
    print('Saved eda_05_community_network.png')
except Exception as _e:
    print('SAVE FAILED eda_05_community_network.png: ' + str(_e))
fig_net.show()

# ---------- Community statistics — VECTORISED intra-ρ ----------
print("\n[COMMUNITY STATISTICS — INEFFICIENT TICKERS ONLY]")
print(f"{'Comm':<6} {'Size':>5} {'Leader':<8} {'Leader Vol':>14} {'Avg Degree':>11} {'Avg Intra-ρ':>12} {'Density':>9}")
print("-" * 75)

for comm_idx, comm in enumerate(sorted(communities, key=len, reverse=True)):
    comm_nodes = sorted(comm)
    leader = community_leaders[comm_idx]
    leader_vol = avg_volume.get(leader, 0)
    subG = G_connected.subgraph(comm_nodes)
    avg_deg = np.mean([G_connected.degree(n) for n in comm_nodes])

    if len(comm_nodes) > 1:
        # >>> VECTORISED: NumPy submatrix + triu_indices instead of nested loop <<<
        valid_members = [t for t in comm_nodes if t in corr_matrix.columns]
        if len(valid_members) > 1:
            sub_corr = corr_matrix.loc[valid_members, valid_members].values
            tri_r, tri_c = np.triu_indices(len(valid_members), k=1)
            avg_intra = float(np.mean(sub_corr[tri_r, tri_c]))
        else:
            avg_intra = 0
        density = nx.density(subG)
    else:
        avg_intra = 1.0
        density = 0.0

    print(f"{comm_idx:<6} {len(comm_nodes):>5} {leader:<8} {leader_vol:>14,.0f} {avg_deg:>11.1f} {avg_intra:>12.3f} {density:>9.3f}")

print(f"\nIsolated inefficient tickers (ρ < {CORR_THRESHOLD} with all others): {len(isolated)}")
if isolated:
    print(f"  {', '.join(sorted(isolated)[:30])}{'...' if len(isolated) > 30 else ''}")

COMMUNITY_LEADERS = community_leaders
print(f"\nStored COMMUNITY_LEADERS: {community_leaders}")

# ============================================================
# DAILY VOLATILITY — VECTORISED earnings matching via merge_asof
# ============================================================
leader_tickers_list = sorted(set(COMMUNITY_LEADERS.values()))
# ============================================================
# HALF-LIFE OF INFORMATION DECAY — VECTORISED crossing detection
# ============================================================
print("\n[14] Half-life of information decay per earnings call — Inefficient Community Leaders...")

WINDOW_POST = 60

halflife_summary = []

for ticker in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == ticker][0]

    tdf = (prices_eda[prices_eda['ticker'] == ticker]
           .copy().sort_values('date').set_index('date'))

    ticker_earnings = (earnings_eda[earnings_eda['ticker'] == ticker]
                       .sort_values('date'))

    for _, erow in ticker_earnings.iterrows():
        edate = erow['date']
        quarter = erow.get('quarter', 'N/A')

        if edate not in tdf.index:
            idx_pos = tdf.index.searchsorted(edate)
            if idx_pos >= len(tdf.index):
                continue
            edate = tdf.index[idx_pos]

        loc = tdf.index.get_loc(edate)
        if isinstance(loc, slice):
            loc = loc.start
        if loc + WINDOW_POST >= len(tdf):
            continue

        pre_price = tdf.iloc[loc]['close']
        post_slice = tdf.iloc[loc: loc + WINDOW_POST + 1]
        car = (post_slice['close'].values - pre_price) / pre_price

        peak_idx = int(np.argmax(np.abs(car)))
        peak_car = car[peak_idx]
        if abs(peak_car) < 1e-6:
            continue

        # >>> VECTORISED: np.argmax on boolean array instead of inner loop <<<
        target = peak_car * 0.5
        half_life = None
        if peak_idx + 1 < len(car):
            remaining = car[peak_idx + 1:]
            if peak_car > 0:
                crossings = remaining <= target
            else:
                crossings = remaining >= target
            if crossings.any():
                half_life = int(np.argmax(crossings)) + 1  # +1 because offset from peak

        halflife_summary.append({
            'ticker': ticker, 'community': comm_idx,
            'date': edate, 'quarter': quarter,
            'half_life': half_life, 'peak_car': peak_car,
            'color': palette[comm_idx % len(palette)]
        })

hl_df = pd.DataFrame(halflife_summary)
hl_df['half_life_val'] = pd.to_numeric(hl_df['half_life'], errors='coerce')

agg = (hl_df.dropna(subset=['half_life_val'])
       .groupby(['ticker', 'community', 'color'])['half_life_val']
       .agg(mean_hl='mean', median_hl='median', n_calls='count')
       .reset_index()
       .sort_values('mean_hl'))

print(f"\n{'='*75}")
print(f"{'Leader':<8} {'Comm':>5} {'N Calls':>8} {'Mean HL (days)':>16} {'Median HL (days)':>18}")
print(f"{'-'*75}")
for _, row in agg.iterrows():
    print(f"{row['ticker']:<8} {int(row['community']):>5} {int(row['n_calls']):>8} "
          f"{row['mean_hl']:>16.1f} {row['median_hl']:>18.1f}")
print(f"{'='*75}")

# ================================================================
# GROUPED BAR CHART — Mean & Median HL per leader
# ================================================================
fig_hl = go.Figure()

fig_hl.add_trace(go.Bar(
    x=agg['ticker'], y=agg['mean_hl'], name='Mean Half-Life',
    marker=dict(color=agg['color'].tolist(), line=dict(color='black', width=1)),
    text=agg['mean_hl'].round(1), textposition='outside', textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>Mean HL: %{y:.1f} days<br>N calls: %{customdata}<extra>Mean</extra>',
    customdata=agg['n_calls']
))
fig_hl.add_trace(go.Bar(
    x=agg['ticker'], y=agg['median_hl'], name='Median Half-Life',
    marker=dict(color=agg['color'].tolist(),
                pattern=dict(shape='/', size=6, solidity=0.4),
                line=dict(color='black', width=1)),
    text=agg['median_hl'].round(1), textposition='outside', textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>Median HL: %{y:.1f} days<br>N calls: %{customdata}<extra>Median</extra>',
    customdata=agg['n_calls']
))

fig_hl.update_layout(
    title=('Information Decay Half-Life per Earnings Call — Inefficient Community Leaders<br>'
           '<sup>Half-life = trading days for post-earnings CAR to retrace 50% of its peak move · '
           f'window = {WINDOW_POST} days · bars sorted by Mean HL · hatched = Median</sup>'),
    xaxis=dict(title='Community Leader (sorted by Mean Half-Life)',
               categoryorder='array', categoryarray=agg['ticker'].tolist(), tickangle=0),
    yaxis=dict(title='Half-Life (Trading Days)', showgrid=True,
               gridcolor='rgba(200,200,200,0.4)', zeroline=True,
               zerolinecolor='black', zerolinewidth=1),
    barmode='group', bargap=0.25, bargroupgap=0.05,
    height=550, width=1200, plot_bgcolor='white',
    legend=dict(orientation='h', y=-0.18, font=dict(size=11)),
    hovermode='x unified'
)
try:
    fig_hl.write_image(EDA_PLOTS_DIR / 'eda_06_half_life_bar.png', scale=2)
    print('Saved eda_06_half_life_bar.png')
except Exception as _e:
    print('SAVE FAILED eda_06_half_life_bar.png: ' + str(_e))
fig_hl.show()

# ================================================================
# PER-CALL SCATTER
# ================================================================
fig_hl_scatter = go.Figure()

for ticker in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == ticker][0]
    color = palette[comm_idx % len(palette)]

    sub = hl_df[(hl_df['ticker'] == ticker) & hl_df['half_life_val'].notna()].copy()
    if sub.empty:
        continue

    fig_hl_scatter.add_trace(go.Scatter(
        x=[ticker] * len(sub), y=sub['half_life_val'], mode='markers',
        marker=dict(size=8, color=color, line=dict(color='black', width=0.8), symbol='circle'),
        name=f'{ticker} (Comm {comm_idx})',
        customdata=list(zip(sub['date'].astype(str), sub['quarter'].astype(str),
                            sub['peak_car'].round(4))),
        hovertemplate=(f'<b>{ticker}</b><br>Date: %{{customdata[0]}}<br>'
                       'Quarter: %{customdata[1]}<br>Peak CAR: %{customdata[2]:.2%}<br>'
                       'Half-Life: %{y:.0f} days<extra></extra>'),
        showlegend=True
    ))

    row_agg = agg[agg['ticker'] == ticker]
    if not row_agg.empty:
        mean_val = row_agg['mean_hl'].values[0]
        median_val = row_agg['median_hl'].values[0]
        for val, label, dash in [(mean_val, 'mean', 'solid'), (median_val, 'median', 'dash')]:
            fig_hl_scatter.add_trace(go.Scatter(
                x=[ticker, ticker], y=[val, val], mode='lines+text',
                line=dict(color=color, width=2.5, dash=dash),
                text=['', f'{label}={val:.0f}d'],
                textposition='middle right', textfont=dict(size=8, color='black'),
                showlegend=False, hoverinfo='skip'
            ))

fig_hl_scatter.update_layout(
    title=('Per-Earnings-Call Half-Life Scatter — Inefficient Community Leaders<br>'
           '<sup>Each point = one earnings call · solid line = mean · dashed = median · '
           f'window = {WINDOW_POST} trading days</sup>'),
    xaxis=dict(title='Community Leader', categoryorder='array',
               categoryarray=agg['ticker'].tolist(), tickangle=0),
    yaxis=dict(title='Half-Life (Trading Days)', showgrid=True,
               gridcolor='rgba(200,200,200,0.4)', zeroline=False),
    height=550, width=1200, plot_bgcolor='white', hovermode='closest',
    legend=dict(title='Leaders', font=dict(size=9), x=1.01, y=1,
                bgcolor='rgba(255,255,255,0.8)')
)
try:
    fig_hl_scatter.write_image(EDA_PLOTS_DIR / 'eda_07_half_life_scatter.png', scale=2)
    print('Saved eda_07_half_life_scatter.png')
except Exception as _e:
    print('SAVE FAILED eda_07_half_life_scatter.png: ' + str(_e))
fig_hl_scatter.show()

# ============================================================
# ============================================================
# SENTIMENT DIVERGENCE: Prepared Remarks vs. Unscripted Q&A (VECTORIZED)
# ============================================================

print("\n[21] Computing Sentiment Divergence for Community Leaders (Dynamic Split)...")

# 1. Filter strictly for Community Leaders
leader_tickers = list(COMMUNITY_LEADERS.values())
div_df = earnings_eda[earnings_eda['ticker'].isin(leader_tickers)].copy()
div_df = div_df.dropna(subset=['transcript']).reset_index(drop=True)

print(f"Analyzing {len(div_df)} total earnings calls for your Community Leaders...")

# 2. Dynamic String Splitting Function
def extract_sections(text):
    # Regex catches variations like "Question-and-Answer Session" or "Q&A Session"
    pattern = re.compile(r'(question-and-answer session|question and answer session|q&a session)', re.IGNORECASE)
    match = pattern.search(text)
    
    if match:
        split_idx = match.start()
        prepared = text[:split_idx]
        qa = text[split_idx:]
        
        # Take the LAST 20000 chars of prepared remarks (usually forward guidance/summary)
        # Take the FIRST 20000 chars of Q&A (usually the most aggressive/important analyst questions)
        return prepared[-20000:], qa[:20000]
    else:
        # Fallback if the specific phrase is missing
        return text[:20000], text[-20000:]

# Apply the split
sections = div_df['transcript'].apply(extract_sections)
prepared_list = [sec[0] for sec in sections]
qa_list = [sec[1] for sec in sections]

# 3. Batch LLM Inference (Fast)
print(" -> Scoring Prepared Remarks (Pre-Split)...")
res_prepared = finbert_pipeline(prepared_list) 

print(" -> Scoring Q&A (Post-Split)...")
res_qa = finbert_pipeline(qa_list)

# 4. Vectorized Post-Processing
def _top(results):
    return [max(r, key=lambda x: x["score"]) if isinstance(r, list) else r for r in results]
df_prepared = pd.DataFrame(_top(res_prepared))
df_qa = pd.DataFrame(_top(res_qa))

label_map = {'positive': 1, 'negative': -1, 'neutral': 0}
prep_multiplier = df_prepared['label'].map(label_map).values
qa_multiplier = df_qa['label'].map(label_map).values

div_df['prepared_sentiment'] = prep_multiplier * df_prepared['score'].values
div_df['qa_sentiment'] = qa_multiplier * df_qa['score'].values

# Vectorized Delta Calculation
div_df['sentiment_delta'] = div_df['qa_sentiment'] - div_df['prepared_sentiment']

# ---------- Visualization ----------
fig_div = go.Figure()

# Add a y=x reference line
fig_div.add_trace(go.Scatter(
    x=[-1, 1], y=[-1, 1],
    mode='lines',
    line=dict(color='black', dash='dash'),
    name='No Tone Change (y=x)'
))

# Colors based on Delta
colors = np.where(div_df['sentiment_delta'] < 0, '#E15759', '#76B7B2')

fig_div.add_trace(go.Scatter(
    x=div_df['prepared_sentiment'],
    y=div_df['qa_sentiment'],
    mode='markers',
    marker=dict(size=9, color=colors, line=dict(width=1, color='darkgray')),
    text=div_df['ticker'],
    customdata=np.column_stack((
        div_df['date'].dt.strftime('%Y-%m-%d'), 
        div_df['sentiment_delta']
    )),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Date: %{customdata[0]}<br>'
        'Prepared Remarks: %{x:.2f}<br>'
        'Q&A Session: %{y:.2f}<br>'
        'Sentiment Delta: %{customdata[1]:.2f}'
        '<extra></extra>'
    ),
    name='Leader Earnings Calls'
))

# Danger Zone
fig_div.add_shape(
    type="rect", x0=0.2, y0=-1.0, x1=1.0, y1=-0.2,
    fillcolor="rgba(225, 87, 89, 0.1)", line=dict(width=0), layer="below"
)
fig_div.add_annotation(
    x=0.6, y=-0.6,
    text="Danger Zone<br>(Management Hiding Bad News)",
    showarrow=False, font=dict(color="#E15759", size=12)
)

fig_div.update_layout(
    title='<b>Sentiment Divergence: Scripted Remarks vs. Unscripted Q&A</b><br>' +
          '<sup>Filtered for Community Leaders | Split dynamically at the Q&A marker</sup>',
    xaxis_title='Prepared Remarks Sentiment (Guidance)',
    yaxis_title='Q&A Sentiment (Analyst Scrutiny)',
    height=650, width=900, plot_bgcolor='white', hovermode='closest'
)

fig_div.add_hline(y=0, line_width=1, line_color="lightgray")
fig_div.add_vline(x=0, line_width=1, line_color="lightgray")

try:
    fig_div.write_image(EDA_PLOTS_DIR / 'eda_08_guidance_qa_divergence.png', scale=2)
    print('Saved eda_08_guidance_qa_divergence.png')
except Exception as _e:
    print('SAVE FAILED eda_08_guidance_qa_divergence.png: ' + str(_e))
fig_div.show()

# ============================================================
# SENTIMENT EXHAUSTION STUDY — VECTORISED price lookups via merge_asof
# ============================================================
print("\n[19] Running Dual-Section Sentiment Exhaustion Study (Sampling Transcripts)...")

exhaustion_data = []
FORWARD_WINDOW = 5

# Build a sorted price lookup table
p_sorted = prices_eda[['ticker', 'date', 'close']].copy().sort_values(['ticker', 'date'])
sample_earnings = earnings_eda.dropna(subset=['transcript']).sample(200, random_state=42)

# Pre-build per-ticker price DataFrames for faster lookups
_price_cache = {}
for t in sample_earnings['ticker'].unique():
    _price_cache[t] = p_sorted[p_sorted['ticker'] == t][['date', 'close']].reset_index(drop=True)

for _, row in tqdm(sample_earnings.iterrows(), total=len(sample_earnings)):
    ticker = row['ticker']
    edate = pd.to_datetime(row['date'])
    full_transcript = row['transcript']

    # --- DYNAMIC SPLIT LOGIC ---
    pattern = re.compile(r'(question-and-answer session|question and answer session|q&a session)', re.IGNORECASE)
    match = pattern.search(full_transcript)
    
    if match:
        split_idx = match.start()
        # Last 5000 chars of prepared (usually forward guidance)
        prepared = full_transcript[:split_idx][-5000:] 
        # First 5000 chars of Q&A (usually the most aggressive analyst questions)
        qa = full_transcript[split_idx:][:5000]
    else:
        # Fallback if marker is missing
        prepared = full_transcript[:5000]
        qa = full_transcript[-5000:]

    # --- SCORING BOTH SECTIONS ---
    try:
        # Score Prepared Remarks
        res_p = finbert_pipeline(prepared, top_k=None)
        probs_p = {r['label']: r['score'] for r in res_p}
        prep_score = probs_p.get('positive', 0) - probs_p.get('negative', 0)
        
        # Score Q&A Session
        res_q = finbert_pipeline(qa, top_k=None)
        probs_q = {r['label']: r['score'] for r in res_q}
        qa_score = probs_q.get('positive', 0) - probs_q.get('negative', 0)
    except Exception:
        continue

    # --- FORWARD RETURN CALCULATION ---
    try:
        tp = _price_cache.get(ticker)
        if tp is None or len(tp) < FORWARD_WINDOW + 2:
            continue

        pos = tp['date'].searchsorted(edate)
        if pos >= len(tp) or pos < 1:
            continue
        if pos + FORWARD_WINDOW >= len(tp):
            continue

        price_t0 = tp.iloc[pos]['close']
        price_t_minus_1 = tp.iloc[pos - 1]['close']
        price_t_plus_5 = tp.iloc[pos + FORWARD_WINDOW]['close']

        day0_ret = (price_t0 - price_t_minus_1) / price_t_minus_1
        forward_5d_ret = (price_t_plus_5 - price_t0) / price_t0

        exhaustion_data.append({
            'ticker': ticker, 
            'date': edate,
            'prepared_sentiment': prep_score,
            'qa_sentiment': qa_score,
            'day0_return': day0_ret,
            'forward_5d_return': forward_5d_ret
        })
    except Exception:
        continue

ex_df = pd.DataFrame(exhaustion_data)
print(f"Collected {len(ex_df)} valid exhaustion data points.")

if ex_df.empty:
    print("No valid exhaustion data — skipping chart.")
else:
    # --- RESHAPE DATA FOR SIDE-BY-SIDE PLOTTING ---
    melted_df = ex_df.melt(
        id_vars=['ticker', 'date', 'day0_return', 'forward_5d_return'],
        value_vars=['prepared_sentiment', 'qa_sentiment'],
        var_name='Section',
        value_name='sentiment'
    )
    
    # Rename for cleaner chart titles
    melted_df['Section'] = melted_df['Section'].replace({
        'prepared_sentiment': 'Prepared Remarks (Structured)',
        'qa_sentiment': 'Q&A Session (Unstructured)'
    })

    # --- PLOTTING ---
    # facet_col creates the side-by-side charts automatically
    fig_ex = px.scatter(
        melted_df, x='sentiment', y='forward_5d_return', color='day0_return',
        facet_col='Section', 
        color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
        hover_data=['ticker', 'date'], trendline="ols"
    )
    
    # Add target lines across ALL subplots
    fig_ex.add_hline(y=0, line_color="black", row='all', col='all')
    fig_ex.add_vline(x=0.8, line_dash="dash", line_color="red", row='all', col='all',
                     annotation_text="Extreme Optimism (>0.8)")

    fig_ex.update_layout(
        title='<b>Sentiment Exhaustion: Structured Script vs. Unscripted Q&A</b><br>' +
              '<sup>Comparing predictive power of management narrative vs. analyst scrutiny on 5-Day Reversal</sup>',
        coloraxis_colorbar=dict(title="Day 0<br>Return"),
        height=600, width=1200, plot_bgcolor='white'
    )
    
    # Clean up axes labels
    fig_ex.update_xaxes(title_text="FinBERT Sentiment Score")
    fig_ex.update_yaxes(title_text="Forward 5-Day Return")

    try:
        fig_ex.write_image(EDA_PLOTS_DIR / 'eda_09_sentiment_exhaustion.png', scale=2)
        print('Saved eda_09_sentiment_exhaustion.png')
    except Exception as _e:
        print('SAVE FAILED eda_09_sentiment_exhaustion.png: ' + str(_e))
    fig_ex.show()

# ============================================================
# SAV ANALYSIS — np.where for bar colours
# ============================================================
print("\n[20] Generating SAV Visualizations for ALL Community Leaders...")

leader_tickers_list = sorted(set(COMMUNITY_LEADERS.values()))

for target_leader in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == target_leader][0]

    tdf = prices_eda[prices_eda['ticker'] == target_leader].copy().sort_values('date')
    tdf['log_ret'] = np.log(tdf['close'] / tdf['close'].shift(1))

    tdf['vol_med20'] = tdf['volume'].rolling(20).median()
    tdf['vol_std20'] = tdf['volume'].rolling(20).std()
    tdf['sav'] = (tdf['volume'] - tdf['vol_med20']) / tdf['vol_std20']
    tdf = tdf.dropna(subset=['sav'])

    raw_e_dates = earnings_eda[earnings_eda['ticker'] == target_leader]['date'].dt.tz_localize(None).dt.normalize()
    tdf['date_only'] = tdf['date'].dt.normalize()

    valid_e_dates = []
    for edate in raw_e_dates:
        pos = tdf['date_only'].searchsorted(edate)
        if pos < len(tdf):
            valid_e_dates.append(tdf['date_only'].iloc[pos])

    tdf['is_earnings'] = tdf['date_only'].isin(valid_e_dates)
    earnings_df = tdf[tdf['is_earnings']]

    fig_sav = make_subplots(
        rows=2, cols=1, row_heights=[0.6, 0.4], vertical_spacing=0.1,
        subplot_titles=(
            f"Time Series: SAV Spikes for ★ {target_leader} (Comm {comm_idx})",
            f"Distribution: Normal Days vs. Earnings Days"
        )
    )

    # >>> VECTORISED: np.where for bar colours <<<
    bar_colors = np.where(tdf['log_ret'] >= 0, '#76B7B2', '#E15759')

    fig_sav.add_trace(go.Bar(
        x=tdf['date'], y=tdf['sav'], marker_color=bar_colors, name='SAV',
        hovertemplate='Date: %{x|%Y-%m-%d}<br>SAV: %{y:.2f}<extra></extra>'
    ), row=1, col=1)

    if not earnings_df.empty:
        fig_sav.add_trace(go.Scatter(
            x=earnings_df['date'], y=earnings_df['sav'] + 0.5, mode='markers',
            marker=dict(symbol='diamond', size=8, color='gold',
                        line=dict(color='black', width=1)),
            name='Earnings Call',
            hovertemplate='<b>Earnings Call</b><br>Date: %{x|%Y-%m-%d}<br>SAV: %{y:.2f}<extra></extra>'
        ), row=1, col=1)

    fig_sav.add_hline(y=2.0, row=1, col=1, line=dict(color='black', dash='dash'),
                      annotation_text="Strategy Trigger (SAV > 2.0)")

    fig_sav.add_trace(go.Histogram(
        x=tdf[~tdf['is_earnings']]['sav'], name='Normal Days',
        marker_color='lightgray', opacity=0.75, histnorm='probability density'
    ), row=2, col=1)

    if not earnings_df.empty:
        fig_sav.add_trace(go.Histogram(
            x=earnings_df['sav'], name='Earnings Days',
            marker_color='gold', opacity=0.75, histnorm='probability density'
        ), row=2, col=1)

    fig_sav.update_layout(
        title=f'<b>Institutional Conviction Filter (SAV) — {target_leader}</b><br><sup>Validating the >2.0 Volume Threshold</sup>',
        height=800, width=1200, plot_bgcolor='white', barmode='overlay', showlegend=True
    )
    fig_sav.update_yaxes(title_text="SAV Score", row=1, col=1)
    fig_sav.update_xaxes(title_text="Standardized Abnormal Volume", row=2, col=1)
    fig_sav.update_yaxes(title_text="Density", row=2, col=1)
    try:
        fig_sav.write_image(EDA_PLOTS_DIR / 'eda_10_sav_filter.png', scale=2)
        print('Saved eda_10_sav_filter.png')
    except Exception as _e:
        print('SAVE FAILED eda_10_sav_filter.png: ' + str(_e))
    fig_sav.show()

# Enchanced EDA - Hypotheses

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import ttest_ind
from pathlib import Path
EDA_PLOTS_DIR = Path('eda_plots')
EDA_PLOTS_DIR.mkdir(exist_ok=True)


POSITIVE_WORDS = {
    "growth",
    "strong",
    "profit",
    "success",
    "opportunity",
    "excellent",
    "positive",
    "gain",
    "improvement",
    "best",
    "higher",
    "increase",
    "rose",
    "record",
    "benefit",
    "up",
    "expanding",
    "revenue",
    "income",
    "outperform",
}

NEGATIVE_WORDS = {
    "loss",
    "decline",
    "negative",
    "risk",
    "fail",
    "difficult",
    "challenging",
    "weak",
    "worst",
    "uncertainty",
    "lower",
    "decrease",
    "fell",
    "drop",
    "down",
    "missed",
    "adverse",
    "struggle",
    "concerns",
    "volatile",
}


# Expected to be defined by the parent script / notebook:
# prices_dev, prices_val, earnings_dev, earnings_val


def calculate_sentiment(text):
    if not isinstance(text, str):
        return 0.0

    words = text.lower().split()
    pos_count = sum(1 for word in words if word in POSITIVE_WORDS)
    neg_count = sum(1 for word in words if word in NEGATIVE_WORDS)
    total = pos_count + neg_count

    if total == 0:
        return 0.0

    return (pos_count - neg_count) / total


def print_section(title):
    print(f"\n{'=' * 80}")
    print(title)
    print(f"{'=' * 80}")


def run_low_vs_high_volatility_anomaly():
    print_section("Hypothesis 1: Low vs High Volatility Anomaly")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    print("Preprocessing...")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])
    df["daily_ret"] = df.groupby("ticker")["close"].pct_change()

    print("Calculating rolling volatility...")
    df["volatility"] = df.groupby("ticker")["daily_ret"].transform(
        lambda series: series.rolling(window=252, min_periods=200).std() * np.sqrt(252)
    )

    print("Resampling to monthly frequency...")
    df = df.set_index("date")

    try:
        monthly_df = df.groupby("ticker").resample("ME").agg({"close": "last", "volatility": "last"})
    except ValueError:
        monthly_df = df.groupby("ticker").resample("M").agg({"close": "last", "volatility": "last"})

    monthly_df = monthly_df.reset_index()
    monthly_df["fwd_ret"] = monthly_df.groupby("ticker")["close"].pct_change().shift(-1)
    monthly_df = monthly_df.dropna(subset=["volatility", "fwd_ret"])

    if monthly_df.empty:
        print("No data available after processing.")
        return

    print("Ranking stocks into deciles...")
    monthly_df["decile"] = monthly_df.groupby("date")["volatility"].transform(
        lambda series: pd.qcut(series, 10, labels=False, duplicates="drop") + 1
        if len(series) >= 20
        else np.nan
    )
    monthly_df = monthly_df.dropna(subset=["decile"])

    print("Computing portfolio stats...")
    portfolio_ts = monthly_df.groupby(["date", "decile"])["fwd_ret"].mean().reset_index()
    stats_df = portfolio_ts.groupby("decile")["fwd_ret"].agg(["mean", "std", "count"])

    stats_df["ann_ret"] = stats_df["mean"] * 12
    stats_df["ann_vol"] = stats_df["std"] * np.sqrt(12)
    stats_df["sharpe"] = stats_df["ann_ret"] / stats_df["ann_vol"]

    print("\n--- Low vs High Volatility Anomaly Results ---")
    print(stats_df[["ann_ret", "ann_vol", "sharpe"]])

    try:
        low_vol_sharpe = stats_df.loc[1.0, "sharpe"]
        high_vol_sharpe = stats_df.loc[10.0, "sharpe"]

        print(f"\nLow Volatility (Decile 1) Sharpe: {low_vol_sharpe:.4f}")
        print(f"High Volatility (Decile 10) Sharpe: {high_vol_sharpe:.4f}")

        if low_vol_sharpe > high_vol_sharpe:
            print("\nConclusion: Low Volatility Anomaly CONFIRMED (Low Vol > High Vol).")
        else:
            print("\nConclusion: Low Volatility Anomaly REJECTED (High Vol > Low Vol).")
    except KeyError:
        print("\nCould not extract Decile 1 or 10 statistics.")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    stats_df["sharpe"].plot(kind="bar", ax=ax1, color="skyblue", edgecolor="black")
    ax1.set_title("Sharpe Ratio by Volatility Decile")
    ax1.set_xlabel("Volatility Decile (1=Lowest, 10=Highest)")
    ax1.set_ylabel("Annualized Sharpe Ratio")
    ax1.grid(axis="y", alpha=0.3)

    ax2.scatter(stats_df["ann_vol"], stats_df["ann_ret"], c="blue", s=100, alpha=0.7)
    ax2.plot(stats_df["ann_vol"], stats_df["ann_ret"], "b--", alpha=0.3)

    for idx, row in stats_df.iterrows():
        ax2.annotate(
            f"D{int(idx)}",
            (row["ann_vol"], row["ann_ret"]),
            xytext=(5, 5),
            textcoords="offset points",
        )

    ax2.set_title("Risk-Return Profile")
    ax2.set_xlabel("Annualized Volatility")
    ax2.set_ylabel("Annualized Return")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h01_low_vs_high_vol_anomaly.png', dpi=150, bbox_inches='tight')
        print('Saved h01_low_vs_high_vol_anomaly.png')
    except Exception as _e:
        print('SAVE FAILED h01_low_vs_high_vol_anomaly.png: ' + str(_e))
    plt.show()


def run_volume_confirmation_experiment():
    print_section("Hypothesis 2: High-Volume vs Low-Volume Up Days")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()
    print("Dataset loaded successfully.")

    df.sort_values(by=["ticker", "date"], inplace=True)
    df["prev_close"] = df.groupby("ticker")["close"].shift(1)
    df["return"] = (df["close"] - df["prev_close"]) / df["prev_close"]
    df["next_return"] = df.groupby("ticker")["return"].shift(-1)
    df["vol_ma_20"] = df.groupby("ticker")["volume"].transform(lambda series: series.rolling(window=20).mean())

    df_clean = df.dropna(subset=["return", "next_return", "vol_ma_20", "volume"]).copy()
    up_days = df_clean[df_clean["return"] > 0].copy()

    print(f"Total Up Days processed: {len(up_days)}")

    high_vol_mask = up_days["volume"] > (1.5 * up_days["vol_ma_20"])
    low_vol_mask = up_days["volume"] <= up_days["vol_ma_20"]

    high_vol_returns = up_days.loc[high_vol_mask, "next_return"]
    low_vol_returns = up_days.loc[low_vol_mask, "next_return"]

    n_high = len(high_vol_returns)
    n_low = len(low_vol_returns)

    if n_high == 0 or n_low == 0:
        print("Insufficient data points for one of the groups.")
        return

    mean_high = high_vol_returns.mean()
    mean_low = low_vol_returns.mean()
    std_high = high_vol_returns.std()
    std_low = low_vol_returns.std()

    print("\n--- Results ---")
    print(
        f"High Volume Up-Days (>150% MA) (n={n_high}): Mean Next-Day Return = "
        f"{mean_high:.6f} ({mean_high * 100:.4f}%), Std = {std_high:.6f}"
    )
    print(
        f"Low Volume Up-Days  (<=100% MA) (n={n_low}): Mean Next-Day Return = "
        f"{mean_low:.6f} ({mean_low * 100:.4f}%), Std = {std_low:.6f}"
    )

    t_stat, p_val = stats.ttest_ind(high_vol_returns, low_vol_returns, equal_var=False)

    print("\n--- T-Test (Welch's) ---")
    print(f"T-Statistic: {t_stat:.4f}")
    print(f"P-Value: {p_val:.4e}")

    if p_val < 0.05:
        if t_stat > 0:
            print("Result: Significant Positive Difference (High Vol > Low Vol).")
        else:
            print("Result: Significant Negative Difference (High Vol < Low Vol).")
    else:
        print("Result: No Statistically Significant Difference.")

    plt.figure(figsize=(10, 6))
    means = [mean_high * 100, mean_low * 100]
    standard_errors = [std_high * 100 / np.sqrt(n_high), std_low * 100 / np.sqrt(n_low)]
    labels = ["High Volume (>150% MA)", "Low Volume (<=100% MA)"]

    plt.bar(labels, means, yerr=standard_errors, capsize=10, color=["#d62728", "#1f77b4"], alpha=0.8)
    plt.title("Mean Next-Day Return: High Vol vs Low Vol Up-Days")
    plt.ylabel("Average Next-Day Return (%)")
    plt.grid(axis="y", linestyle="--", alpha=0.5)

    for idx, value in enumerate(means):
        plt.text(idx, value + (0.01 if value >= 0 else -0.05), f"{value:.4f}%", ha="center", fontweight="bold")

    try:
        plt.savefig(EDA_PLOTS_DIR / 'h02_volume_confirmation.png', dpi=150, bbox_inches='tight')
        print('Saved h02_volume_confirmation.png')
    except Exception as _e:
        print('SAVE FAILED h02_volume_confirmation.png: ' + str(_e))
    plt.show()


def run_rsi_mean_reversion_experiment():
    print_section("Hypothesis 3: RSI Oversold vs Overbought")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    print("Calculating RSI...")
    df["delta"] = df.groupby("ticker")["close"].diff()

    up = df["delta"].clip(lower=0)
    down = -1 * df["delta"].clip(upper=0)

    ma_up = up.groupby(df["ticker"]).ewm(alpha=1 / 14, adjust=False).mean()
    ma_down = down.groupby(df["ticker"]).ewm(alpha=1 / 14, adjust=False).mean()

    df["avg_gain"] = ma_up.reset_index(level=0, drop=True)
    df["avg_loss"] = ma_down.reset_index(level=0, drop=True)

    with np.errstate(divide="ignore", invalid="ignore"):
        rs = df["avg_gain"] / df["avg_loss"]
        df["rsi"] = 100 - (100 / (1 + rs))

    df.loc[df["avg_loss"] == 0, "rsi"] = 100
    df["next_close"] = df.groupby("ticker")["close"].shift(-1)
    df["next_ret"] = (df["next_close"] - df["close"]) / df["close"]

    valid_data = df.dropna(subset=["rsi", "next_ret"])
    oversold = valid_data[valid_data["rsi"] < 30]["next_ret"]
    overbought = valid_data[valid_data["rsi"] > 70]["next_ret"]

    print(f"Oversold (RSI < 30) count: {len(oversold)}")
    print(f"Overbought (RSI > 70) count: {len(overbought)}")

    mean_os = oversold.mean()
    mean_ob = overbought.mean()

    print(f"Mean Next-Day Return (Oversold): {mean_os:.6f}")
    print(f"Mean Next-Day Return (Overbought): {mean_ob:.6f}")

    t_stat, p_val = ttest_ind(oversold, overbought, equal_var=False)
    print(f"T-statistic: {t_stat:.4f}")
    print(f"P-value: {p_val:.4e}")

    plt.figure(figsize=(10, 6))
    plt.hist(oversold, bins=100, range=(-0.05, 0.05), density=True, alpha=0.5, label="Oversold (RSI < 30)")
    plt.hist(overbought, bins=100, range=(-0.05, 0.05), density=True, alpha=0.5, label="Overbought (RSI > 70)")
    plt.title("Next-Day Return Distribution: Oversold vs Overbought")
    plt.xlabel("Next Day Return")
    plt.ylabel("Density")
    plt.legend()
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h03_rsi_mean_reversion.png', dpi=150, bbox_inches='tight')
        print('Saved h03_rsi_mean_reversion.png')
    except Exception as _e:
        print('SAVE FAILED h03_rsi_mean_reversion.png: ' + str(_e))
    plt.show()


def run_illiquidity_premium_experiment():
    print_section("Hypothesis 4: Illiquidity Premium")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])

    print("Calculating Amihud Ratio...")
    df["ret"] = df.groupby("ticker")["close"].pct_change()
    df["dollar_vol"] = (df["close"] * df["volume"]).replace(0, np.nan)
    df["amihud"] = (df["ret"].abs() / df["dollar_vol"]).replace([np.inf, -np.inf], np.nan)

    print("Aggregating to monthly frequency...")
    monthly_data = df.groupby(["ticker", pd.Grouper(key="date", freq="ME")]).agg({"amihud": "mean", "close": "last"}).reset_index()
    monthly_data["ret_m"] = monthly_data.groupby("ticker")["close"].pct_change()
    monthly_data["fwd_ret_m"] = monthly_data.groupby("ticker")["ret_m"].shift(-1)

    valid_data = monthly_data.dropna(subset=["amihud", "fwd_ret_m"]).copy()

    print("Ranking stocks into deciles...")

    def rank_deciles(series):
        try:
            return pd.qcut(series, 10, labels=False, duplicates="drop")
        except ValueError:
            return np.nan

    valid_data["decile"] = valid_data.groupby("date")["amihud"].transform(rank_deciles)
    valid_data = valid_data.dropna(subset=["decile"])

    print("Constructing portfolios...")
    portfolios = valid_data.groupby(["date", "decile"])["fwd_ret_m"].mean().unstack()

    if 0.0 not in portfolios.columns or 9.0 not in portfolios.columns:
        print("Error: Deciles 0 and 9 not found in aggregated data.")
        print("Columns found:", portfolios.columns)
        return

    illiquid_returns = portfolios[9.0]
    liquid_returns = portfolios[0.0]
    spread_returns = illiquid_returns - liquid_returns

    mean_spread = spread_returns.mean()
    t_stat, p_val = stats.ttest_1samp(spread_returns, 0)

    print("\n--- Illiquidity Premium Analysis Results ---")
    print(f"Observation Period: {valid_data['date'].min().date()} to {valid_data['date'].max().date()}")
    print(f"Total Months: {len(spread_returns)}")
    print(f"Average Monthly Spread Return: {mean_spread:.4%}")
    print(f"Annualized Spread Return: {(1 + mean_spread) ** 12 - 1:.4%}")
    print(f"T-Statistic: {t_stat:.4f}")
    print(f"P-Value: {p_val:.4f}")

    cum_illiquid = (1 + illiquid_returns).cumprod()
    cum_liquid = (1 + liquid_returns).cumprod()
    cum_spread = (1 + spread_returns).cumprod()

    plt.figure(figsize=(10, 6))
    plt.plot(cum_illiquid.index, cum_illiquid, label="Illiquid (Top Decile)")
    plt.plot(cum_liquid.index, cum_liquid, label="Liquid (Bottom Decile)")
    plt.plot(cum_spread.index, cum_spread, label="Long Illiquid / Short Liquid", linestyle="--", color="black")
    plt.title("Illiquidity Premium: Cumulative Returns")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h04_illiquidity_premium.png', dpi=150, bbox_inches='tight')
        print('Saved h04_illiquidity_premium.png')
    except Exception as _e:
        print('SAVE FAILED h04_illiquidity_premium.png: ' + str(_e))
    plt.show()


def run_sentiment_acceleration_experiment():
    print_section("Hypothesis 5: Sentiment Acceleration vs Post-Earnings Sharpe")
    print("Using preloaded earnings_dev, earnings_val, prices_dev, and prices_val...")
    df_earnings = pd.concat([earnings_dev, earnings_val], ignore_index=True)
    df_prices = pd.concat([prices_dev, prices_val], ignore_index=True)

    print(f"Earnings events loaded: {len(df_earnings)}")
    print(f"Price records loaded: {len(df_prices)}")

    print("Calculating sentiment scores...")
    df_earnings["sentiment"] = df_earnings["transcript"].apply(calculate_sentiment)

    print("Calculating sentiment acceleration...")
    df_earnings["date"] = pd.to_datetime(df_earnings["date"]).dt.tz_localize(None).dt.normalize()
    df_earnings = df_earnings.sort_values(by=["ticker", "date"])
    df_earnings["prev_sentiment"] = df_earnings.groupby("ticker")["sentiment"].shift(1)
    df_earnings["sentiment_change"] = df_earnings["sentiment"] - df_earnings["prev_sentiment"]
    df_earnings_clean = df_earnings.dropna(subset=["sentiment_change"]).copy()

    print("Processing price data...")
    df_prices["date"] = pd.to_datetime(df_prices["date"]).dt.tz_localize(None).dt.normalize()
    df_prices = df_prices.sort_values(["ticker", "date"])
    df_prices["return"] = df_prices.groupby("ticker")["close"].pct_change()

    print("Building price lookup dictionary...")
    price_dict = {ticker: frame.set_index("date")["return"] for ticker, frame in df_prices.groupby("ticker")}

    sharpe_ratios = []
    valid_indices = []

    print(f"Calculating post-earnings Sharpe Ratio for {len(df_earnings_clean)} events...")

    for idx, row in df_earnings_clean.iterrows():
        ticker = row["ticker"]
        earnings_date = row["date"]

        if ticker not in price_dict:
            continue

        ticker_returns = price_dict[ticker]

        try:
            idx_loc = ticker_returns.index.searchsorted(earnings_date, side="left")

            if idx_loc >= len(ticker_returns):
                continue

            current_date_at_idx = ticker_returns.index[idx_loc]
            start_pos = idx_loc + 1 if current_date_at_idx == earnings_date else idx_loc
            end_pos = start_pos + 20

            if end_pos > len(ticker_returns):
                continue

            window_returns = ticker_returns.iloc[start_pos:end_pos].values
            window_returns = window_returns[~np.isnan(window_returns)]

            if len(window_returns) < 10:
                continue

            mean_ret = np.mean(window_returns)
            std_ret = np.std(window_returns)
            sharpe = 0 if std_ret == 0 or np.isnan(std_ret) else mean_ret / std_ret

            sharpe_ratios.append(sharpe)
            valid_indices.append(idx)
        except Exception:
            continue

    df_analysis = df_earnings_clean.loc[valid_indices].copy()
    df_analysis["sharpe_ratio"] = sharpe_ratios

    print(f"Successfully computed Sharpe Ratios for {len(df_analysis)} events.")

    if len(df_analysis) < 50:
        print("Insufficient data points for meaningful regression.")
        return

    print("\nRunning Regression Analysis...")
    df_analysis = df_analysis.dropna(subset=["sharpe_ratio", "sentiment", "sentiment_change"])
    df_analysis["sentiment_z"] = (df_analysis["sentiment"] - df_analysis["sentiment"].mean()) / df_analysis["sentiment"].std()
    df_analysis["sentiment_change_z"] = (
        (df_analysis["sentiment_change"] - df_analysis["sentiment_change"].mean()) / df_analysis["sentiment_change"].std()
    )

    X1 = sm.add_constant(df_analysis["sentiment_z"])
    y = df_analysis["sharpe_ratio"]
    model1 = sm.OLS(y, X1).fit()

    X2 = sm.add_constant(df_analysis["sentiment_change_z"])
    model2 = sm.OLS(y, X2).fit()

    X3 = sm.add_constant(df_analysis[["sentiment_z", "sentiment_change_z"]])
    model3 = sm.OLS(y, X3).fit()

    print("\n=== Regression 1: Sharpe ~ Absolute Sentiment (Z-scored) ===")
    print(model1.summary().tables[1])
    print(f"R-squared: {model1.rsquared:.6f}")

    print("\n=== Regression 2: Sharpe ~ Sentiment Change (Z-scored) ===")
    print(model2.summary().tables[1])
    print(f"R-squared: {model2.rsquared:.6f}")

    print("\n=== Regression 3: Sharpe ~ Absolute + Change ===")
    print(model3.summary().tables[1])
    print(f"R-squared: {model3.rsquared:.6f}")

    df_analysis["group"] = df_analysis["sentiment_change"].apply(lambda value: "Improving" if value > 0 else "Deteriorating")
    group_means = df_analysis.groupby("group")["sharpe_ratio"].mean()
    group_counts = df_analysis.groupby("group")["sharpe_ratio"].count()

    print("\n=== Group Comparison (Average Daily Sharpe Ratio) ===")
    print(group_means)
    print("\nGroup Counts:")
    print(group_counts)

    plt.figure(figsize=(10, 6))
    group_means.plot(kind="bar", color=["red", "green"], alpha=0.7)
    plt.title("Average Post-Earnings Sharpe Ratio (t+1 to t+20)\nby Sentiment Momentum")
    plt.ylabel("Average Daily Sharpe Ratio")
    plt.xlabel("Sentiment Change")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h05_sentiment_acceleration.png', dpi=150, bbox_inches='tight')
        print('Saved h05_sentiment_acceleration.png')
    except Exception as _e:
        print('SAVE FAILED h05_sentiment_acceleration.png: ' + str(_e))
    plt.show()


def run_volatility_acceleration_experiment():
    print_section("Hypothesis 6: Volatility Acceleration vs Forward Drawdown")
    df = pd.concat([prices_dev, prices_val], ignore_index=True)
    df.sort_values(["ticker", "date"], inplace=True)

    df["ret"] = df.groupby("ticker")["close"].pct_change()
    df["vol_10"] = df.groupby("ticker")["ret"].transform(lambda series: series.rolling(10, min_periods=10).std())
    df["vol_60"] = df.groupby("ticker")["ret"].transform(lambda series: series.rolling(60, min_periods=60).std())
    df["vol_ratio"] = df["vol_10"] / df["vol_60"]

    df["next_low"] = df.groupby("ticker")["low"].shift(-1)

    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=20)
    df["fwd_min_low"] = df.groupby("ticker")["next_low"].transform(
        lambda series: series.rolling(window=indexer, min_periods=20).min()
    )
    df["fwd_drawdown"] = (df["fwd_min_low"] - df["close"]) / df["close"]

    valid_data = df.dropna(subset=["vol_ratio", "fwd_drawdown"])
    shock_group = valid_data[valid_data["vol_ratio"] > 1.5]["fwd_drawdown"]
    control_group = valid_data[(valid_data["vol_ratio"] >= 0.9) & (valid_data["vol_ratio"] <= 1.1)]["fwd_drawdown"]

    ks_stat, p_value = stats.ks_2samp(shock_group, control_group)

    print("=== Volatility Acceleration Experiment Results ===")
    print(f"Shock Group Size: {len(shock_group)}")
    print(f"Control Group Size: {len(control_group)}")
    print(f"Shock Group Mean Forward Drawdown (Next 20 Days): {shock_group.mean():.4f}")
    print(f"Control Group Mean Forward Drawdown (Next 20 Days): {control_group.mean():.4f}")
    print(f"KS Test Statistic: {ks_stat:.4f}")
    print(f"KS Test p-value: {p_value:.4e}")

    shock_prob = (shock_group < -0.10).mean()
    control_prob = (control_group < -0.10).mean()
    print(f"Probability of >10% price drop in Shock group: {shock_prob:.2%}")
    print(f"Probability of >10% price drop in Control group: {control_prob:.2%}")

    plt.figure(figsize=(10, 6))
    plt.hist(control_group, bins=100, range=(-0.5, 0.1), density=True, alpha=0.5, label="Control (Normal Vol)", color="blue")
    plt.hist(shock_group, bins=100, range=(-0.5, 0.1), density=True, alpha=0.5, label="Shock (Vol Accel > 1.5x)", color="red")
    plt.title("Distribution of Maximum Forward Drawdown (Next 20 Days)")
    plt.xlabel("Max Drawdown (Relative to Entry Price)")
    plt.ylabel("Density")
    plt.axvline(-0.10, color="black", linestyle="--", alpha=0.7, label="10% Drop Threshold")
    plt.legend()
    plt.grid(True, alpha=0.3)
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h06_volatility_acceleration.png', dpi=150, bbox_inches='tight')
        print('Saved h06_volatility_acceleration.png')
    except Exception as _e:
        print('SAVE FAILED h06_volatility_acceleration.png: ' + str(_e))
    plt.show()


def run_trend_filter_crash_experiment():
    print_section("Hypothesis 7: Trend Filter During Crash Periods")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])

    print("Calculating SMA and Returns...")
    grouped = df.groupby("ticker")
    df["sma_200"] = grouped["close"].transform(lambda series: series.rolling(200).mean())
    df["prev_close"] = grouped["close"].shift(1)
    df["return"] = df["close"] / df["prev_close"] - 1.0
    df["signal"] = (df["close"] > df["sma_200"]).astype(int)
    df["strategy_pos"] = grouped["signal"].shift(1).fillna(0)
    df["strat_ret"] = df["return"] * df["strategy_pos"]

    market_series = df.groupby("date")["close"].mean()
    market_peak = market_series.cummax()
    market_dd = (market_series - market_peak) / market_peak

    global_low_date = market_dd.idxmin()
    peak_date = market_series.loc[:global_low_date].idxmax()

    crash_start = peak_date
    crash_end = global_low_date
    print(f"Crash Period Identified: {crash_start.date()} to {crash_end.date()}")
    print(f"Market Drawdown: {market_dd.min():.2%}")

    lookback_start = crash_start - pd.Timedelta(days=180)
    pre_crash_data = df[(df["date"] >= lookback_start) & (df["date"] < crash_start)]
    vol_stats = pre_crash_data.groupby("ticker")["return"].std()

    vol_threshold = vol_stats.quantile(0.8)
    high_vol_tickers = vol_stats[vol_stats >= vol_threshold].index
    print(f"High Volatility Threshold (Std Dev): {vol_threshold:.4f}")
    print(f"Selected {len(high_vol_tickers)} high volatility stocks.")

    sim_data = df[
        (df["date"] >= crash_start)
        & (df["date"] <= crash_end)
        & (df["ticker"].isin(high_vol_tickers))
    ].copy()

    results = []

    for ticker, group in sim_data.groupby("ticker"):
        if len(group) < 10:
            continue

        bh_equity = (1 + group["return"].fillna(0)).cumprod()
        bh_peak = bh_equity.cummax()
        bh_dd_series = (bh_equity - bh_peak) / bh_peak
        bh_mdd = bh_dd_series.min()

        tr_equity = (1 + group["strat_ret"].fillna(0)).cumprod()
        tr_peak = tr_equity.cummax()
        tr_dd_series = (tr_equity - tr_peak) / tr_peak
        tr_mdd = tr_dd_series.min()

        results.append({"ticker": ticker, "BH_MDD": bh_mdd, "Trend_MDD": tr_mdd})

    results_df = pd.DataFrame(results).dropna()

    mean_bh_mdd = results_df["BH_MDD"].mean()
    mean_tr_mdd = results_df["Trend_MDD"].mean()

    print("\n=== Simulation Results ===")
    print(f"Average Max Drawdown (Buy & Hold): {mean_bh_mdd:.2%}")
    print(f"Average Max Drawdown (Trend Filter): {mean_tr_mdd:.2%}")

    improvement = mean_tr_mdd - mean_bh_mdd
    print(f"Average Improvement: {improvement * 100:.2f} percentage points")

    t_stat, p_val = stats.ttest_rel(results_df["BH_MDD"], results_df["Trend_MDD"])
    print(f"Paired t-test: t-statistic={t_stat:.4f}, p-value={p_val:.4e}")

    if p_val < 0.05:
        print("Result: Statistically significant difference.")
    else:
        print("Result: No statistically significant difference.")

    plt.figure(figsize=(8, 6))
    plt.boxplot([results_df["BH_MDD"], results_df["Trend_MDD"]], tick_labels=["Buy & Hold", "Trend Filter"])
    plt.title("Distribution of Maximum Drawdowns (High Volatility Stocks)")
    plt.ylabel("Maximum Drawdown")
    plt.grid(True, axis="y", alpha=0.3)
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h07_trend_filter_drawdown.png', dpi=150, bbox_inches='tight')
        print('Saved h07_trend_filter_drawdown.png')
    except Exception as _e:
        print('SAVE FAILED h07_trend_filter_drawdown.png: ' + str(_e))
    plt.show()

    piv_bh = sim_data.pivot(index="date", columns="ticker", values="return").mean(axis=1).fillna(0)
    piv_tr = sim_data.pivot(index="date", columns="ticker", values="strat_ret").mean(axis=1).fillna(0)

    cum_bh = (1 + piv_bh).cumprod()
    cum_tr = (1 + piv_tr).cumprod()

    plt.figure(figsize=(12, 6))
    plt.plot(cum_bh, label="Buy & Hold Portfolio", color="red")
    plt.plot(cum_tr, label="Trend Filter Portfolio", color="blue")
    plt.title(f"Performance of High Volatility Stocks during Crash ({crash_start.date()} - {crash_end.date()})")
    plt.xlabel("Date")
    plt.ylabel("Normalized Equity")
    plt.legend()
    plt.grid(True, alpha=0.3)
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h07_trend_filter_equity.png', dpi=150, bbox_inches='tight')
        print('Saved h07_trend_filter_equity.png')
    except Exception as _e:
        print('SAVE FAILED h07_trend_filter_equity.png: ' + str(_e))
    plt.show()

def run_parkinson_volatility_persistence_experiment():
    print_section("Hypothesis 9: Parkinson Volatility vs Historical Volatility Persistence")
    df = prices_dev.copy()

    df = df.copy()
    df.sort_values(by=["ticker", "date"], inplace=True)

    df["prev_close"] = df.groupby("ticker")["close"].shift(1)
    df["log_ret"] = np.log(df["close"] / df["prev_close"])

    df["hist_vol"] = df.groupby("ticker")["log_ret"].transform(lambda series: series.rolling(window=20).std())

    df["log_hl_sq"] = (np.log(df["high"] / df["low"])) ** 2
    const_factor = 1.0 / (4.0 * np.log(2.0))
    df["park_vol_sq"] = df.groupby("ticker")["log_hl_sq"].transform(lambda series: series.rolling(window=20).mean())
    df["park_vol"] = np.sqrt(const_factor * df["park_vol_sq"])

    df["future_vol"] = df.groupby("ticker")["hist_vol"].shift(-20)
    valid_df = df.dropna(subset=["hist_vol", "park_vol", "future_vol"])

    print(f"Data points available for correlation analysis: {len(valid_df)}")
    if len(valid_df) == 0:
        print("No valid data points found.")
        return

    corr_hv = valid_df["hist_vol"].corr(valid_df["future_vol"])
    corr_pv = valid_df["park_vol"].corr(valid_df["future_vol"])

    print("\n--- Correlation Results ---")
    print(f"Historical Volatility vs Future Realized Volatility: {corr_hv:.4f}")
    print(f"Parkinson Volatility vs Future Realized Volatility:  {corr_pv:.4f}")

    print("\n--- Conclusion ---")
    if corr_pv > corr_hv:
        print("Parkinson Volatility has higher predictive persistence (higher correlation with future volatility).")
    else:
        print("Historical Volatility has higher predictive persistence (higher correlation with future volatility).")

    plot_df = valid_df.sample(min(5000, len(valid_df)), random_state=42)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Hypothesis 9: Parkinson vs Historical Volatility Persistence", fontsize=14, fontweight="bold")

    axes[0].scatter(plot_df["hist_vol"], plot_df["future_vol"], alpha=0.2, s=10, color="#1f77b4")
    axes[0].set_title(f"Historical Vol vs Future Vol\nCorr = {corr_hv:.4f}")
    axes[0].set_xlabel("20-Day Historical Volatility")
    axes[0].set_ylabel("Future Realized Volatility")
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(plot_df["park_vol"], plot_df["future_vol"], alpha=0.2, s=10, color="#d62728")
    axes[1].set_title(f"Parkinson Vol vs Future Vol\nCorr = {corr_pv:.4f}")
    axes[1].set_xlabel("20-Day Parkinson Volatility")
    axes[1].set_ylabel("Future Realized Volatility")
    axes[1].grid(True, alpha=0.3)

    labels = ["Historical Vol", "Parkinson Vol"]
    correlations = [corr_hv, corr_pv]
    colors = ["#1f77b4", "#d62728"]
    axes[2].bar(labels, correlations, color=colors, edgecolor="black", alpha=0.85)
    axes[2].set_title("Correlation With Future Realized Volatility")
    axes[2].set_ylabel("Correlation")
    axes[2].grid(axis="y", alpha=0.3)
    for idx, value in enumerate(correlations):
        axes[2].text(idx, value + 0.005, f"{value:.4f}", ha="center", fontweight="bold")

    plt.tight_layout()
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h08_parkinson_vol_persistence.png', dpi=150, bbox_inches='tight')
        print('Saved h08_parkinson_vol_persistence.png')
    except Exception as _e:
        print('SAVE FAILED h08_parkinson_vol_persistence.png: ' + str(_e))
    plt.show()


def run_amihud_vol_window_search_experiment():
    """
    Hypothesis 8: Amihud & Volatility Lookback Window Grid Search.

    Tests whether the 60-day lookback currently used for both the Amihud
    illiquidity ratio and the volatility metric in universe selection is
    optimal, or whether a shorter/longer window (20, 40, 60, 120 days)
    produces a better-quality investable universe.

    Method:
      For each (amihud_window, vol_window) combination:
        1. Compute rolling Amihud(W) and rolling Vol(W) per ticker.
        2. Monthly: rank tickers by each metric, form composite score,
           select top 40% (universe).
        3. Measure equal-weighted forward 1-month return of that universe.
        4. Compute annualised Sharpe of the strategy.

    A higher Sharpe = better predictive quality from that window.
    """
    print_section("Hypothesis 8: Amihud & Volatility Lookback Window Grid Search")
    print("Using preloaded prices_dev...")

    df = prices_dev.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])

    df["ret"] = df.groupby("ticker")["close"].pct_change()
    df["dollar_vol"] = (df["close"] * df["volume"]).replace(0, np.nan)
    df["amihud_daily"] = (df["ret"].abs() / df["dollar_vol"]).replace([np.inf, -np.inf], np.nan)

    windows = [20, 40, 60, 120]
    results = []

    for amihud_w in windows:
        for vol_w in windows:
            label = f"Amihud-{amihud_w}d / Vol-{vol_w}d"
            print(f"  Testing {label}...")

            # --- Compute windowed metrics for every row ---
            df["amihud_roll"] = (
                df.groupby("ticker")["amihud_daily"]
                .transform(lambda s: s.rolling(amihud_w, min_periods=max(10, amihud_w // 2)).mean())
            )
            df["vol_roll"] = (
                df.groupby("ticker")["ret"]
                .transform(lambda s: s.rolling(vol_w, min_periods=max(10, vol_w // 2)).std() * np.sqrt(252))
            )

            # --- Resample to month-end snapshots ---
            df_indexed = df.set_index("date")
            try:
                monthly = (
                    df_indexed.groupby("ticker")
                    .resample("ME")
                    .agg({"close": "last", "amihud_roll": "last", "vol_roll": "last"})
                )
            except ValueError:
                monthly = (
                    df_indexed.groupby("ticker")
                    .resample("M")
                    .agg({"close": "last", "amihud_roll": "last", "vol_roll": "last"})
                )
            monthly = monthly.reset_index()
            monthly["fwd_ret"] = monthly.groupby("ticker")["close"].pct_change().shift(-1)
            monthly = monthly.dropna(subset=["amihud_roll", "vol_roll", "fwd_ret"])

            # --- Monthly universe selection: top 40% composite score ---
            def score_month(group):
                if len(group) < 5:
                    group["selected"] = False
                    return group
                group = group.copy()
                group["vol_rank"] = group["vol_roll"].rank(ascending=True, method="average")
                group["amihud_rank"] = group["amihud_roll"].rank(ascending=False, method="average")
                group["composite"] = group["vol_rank"] + group["amihud_rank"]
                threshold = group["composite"].quantile(0.60)  # top 40%
                group["selected"] = group["composite"] >= threshold
                return group

            monthly = monthly.groupby("date", group_keys=False).apply(score_month)

            # --- Equal-weighted portfolio return of selected universe ---
            portfolio = (
                monthly[monthly["selected"]]
                .groupby("date")["fwd_ret"]
                .mean()
            )

            if len(portfolio) < 12:
                print(f"    Insufficient months ({len(portfolio)}) — skipping.")
                continue

            ann_ret = portfolio.mean() * 12
            ann_vol = portfolio.std() * np.sqrt(12)
            sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
            hit_rate = (portfolio > 0).mean()

            results.append({
                "label": label,
                "amihud_window": amihud_w,
                "vol_window": vol_w,
                "ann_return": ann_ret,
                "ann_vol": ann_vol,
                "sharpe": sharpe,
                "hit_rate": hit_rate,
                "months": len(portfolio),
            })
            print(f"    Sharpe={sharpe:.3f}  AnnRet={ann_ret:.2%}  AnnVol={ann_vol:.2%}  HitRate={hit_rate:.2%}")

    if not results:
        print("No valid results produced.")
        return

    results_df = pd.DataFrame(results).sort_values("sharpe", ascending=False)

    print("\n--- Grid Search Results (sorted by Sharpe) ---")
    print(results_df[["label", "ann_return", "ann_vol", "sharpe", "hit_rate", "months"]].to_string(index=False))

    best = results_df.iloc[0]
    current = results_df[results_df["label"] == "Amihud-60d / Vol-60d"]
    print(f"\nBest window  : {best['label']} (Sharpe={best['sharpe']:.4f})")
    if not current.empty:
        curr_sharpe = current.iloc[0]["sharpe"]
        print(f"Current (60d): Amihud-60d / Vol-60d (Sharpe={curr_sharpe:.4f})")
        diff = best["sharpe"] - curr_sharpe
        if diff > 0.05:
            print(f"Conclusion: A different window improves Sharpe by {diff:.4f} — consider changing the lookback.")
        else:
            print(f"Conclusion: 60-day window is competitive (difference < 0.05 Sharpe). Current setting is acceptable.")

    # --- Plots ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Hypothesis 8: Amihud & Vol Lookback Window Grid Search", fontsize=14, fontweight="bold")

    # Heatmap: Sharpe by (amihud_w, vol_w)
    pivot_sharpe = results_df.pivot(index="amihud_window", columns="vol_window", values="sharpe")
    im = axes[0].imshow(pivot_sharpe.values, cmap="RdYlGn", aspect="auto")
    axes[0].set_xticks(range(len(pivot_sharpe.columns)))
    axes[0].set_yticks(range(len(pivot_sharpe.index)))
    axes[0].set_xticklabels([f"{c}d" for c in pivot_sharpe.columns])
    axes[0].set_yticklabels([f"{r}d" for r in pivot_sharpe.index])
    axes[0].set_xlabel("Vol Window")
    axes[0].set_ylabel("Amihud Window")
    axes[0].set_title("Sharpe Ratio Heatmap")
    for i in range(len(pivot_sharpe.index)):
        for j in range(len(pivot_sharpe.columns)):
            val = pivot_sharpe.values[i, j]
            if not np.isnan(val):
                axes[0].text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9,
                             color="black" if 0.2 < val < 0.8 else "white")
    fig.colorbar(im, ax=axes[0])

    # Bar: Sharpe by label
    colors = ["#2E86AB" if r["label"] != "Amihud-60d / Vol-60d" else "#E15759" for _, r in results_df.iterrows()]
    axes[1].barh(results_df["label"], results_df["sharpe"], color=colors, edgecolor="black", alpha=0.85)
    axes[1].axvline(x=0, color="black", linewidth=0.8)
    axes[1].set_xlabel("Annualised Sharpe Ratio")
    axes[1].set_title("Sharpe by Window Combination\n(red = current 60d/60d)")
    axes[1].grid(axis="x", alpha=0.3)

    # Scatter: Ann Return vs Ann Vol
    for _, row in results_df.iterrows():
        color = "#E15759" if row["label"] == "Amihud-60d / Vol-60d" else "#2E86AB"
        axes[2].scatter(row["ann_vol"], row["ann_return"], color=color, s=80, zorder=3)
        axes[2].annotate(
            row["label"].replace(" / ", "\n"),
            (row["ann_vol"], row["ann_return"]),
            xytext=(4, 4), textcoords="offset points", fontsize=7
        )
    axes[2].set_xlabel("Annualised Volatility")
    axes[2].set_ylabel("Annualised Return")
    axes[2].set_title("Risk-Return by Window")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    try:
        plt.savefig(EDA_PLOTS_DIR / 'h09_amihud_vol_window_search.png', dpi=150, bbox_inches='tight')
        print('Saved h09_amihud_vol_window_search.png')
    except Exception as _e:
        print('SAVE FAILED h09_amihud_vol_window_search.png: ' + str(_e))
    plt.show()




required_datasets = ("prices_dev", "prices_val", "earnings_dev", "earnings_val")
missing = [name for name in required_datasets if name not in globals()]
if missing:
    raise NameError(
        "Missing preloaded dataset variables: "
        + ", ".join(missing)
        + ". Define prices_dev, prices_val, earnings_dev, and earnings_val before running this file."
    )

experiments = [
    run_low_vs_high_volatility_anomaly,
    run_volume_confirmation_experiment,
    run_rsi_mean_reversion_experiment,
    run_illiquidity_premium_experiment,
    run_sentiment_acceleration_experiment,
    run_volatility_acceleration_experiment,
    run_trend_filter_crash_experiment,
    run_amihud_vol_window_search_experiment,
    run_parkinson_volatility_persistence_experiment,
]

for experiment in experiments:
    try:
        experiment()
    except Exception as exc:
        print(f"Experiment '{experiment.__name__}' failed: {exc}")

